In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Recursive Set

This notebook implements a `RecursiveSet` data structure in TypeScript. Unlike standard JavaScript `Set` objects which rely on object identity (references), this implementation enforces **structural equality**.

Two sets $A$ and $B$ are considered equal ($A = B$) iff they contain the same elements, regardless of the order in which they were added or their memory addresses. This property allows `RecursiveSet` to be used as elements within other `RecursiveSet` instances, enabling the construction of recursive mathematical structures like $\mathcal{P}(\mathcal{P}(\mathbb{N}))$.

The implementation relies on:
1.  **Canonical Ordering**: Elements are stored internally in a sorted array `_elements`.
2.  **Hashing**: A 32-bit integer hash is lazily computed and cached to speed up equality checks.

## FNV-1a Hash Function

The function `hashString` computes a hash value for a given string $s$. It implements the **FNV-1a** (Fowler–Noll–Vo) algorithm, which is designed for fast hashing with good dispersion properties.

**Input:**
*   `str`: The string $s$ to be hashed.

**Output:**
*   A 32-bit unsigned integer representing the hash code.

**Mathematical Sketch:**
Let $s$ be a sequence of bytes $b_0, \dots, b_{n-1}$. The hash $H$ is defined recursively using 32-bit arithmetic modulo $2^{32}$.

We define the constants for the 32-bit FNV-1a algorithm:
*   $\text{offset_basis} = \texttt{0x811C9DC5}$
*   $\text{prime} = \texttt{0x01000193}$

The recursion is:
$$
\begin{aligned}
H_0 &= \texttt{0x811C9DC5} \\
H_{i+1} &= (H_i \oplus b_i) \cdot \texttt{0x01000193} \pmod{2^{32}}
\end{aligned}
$$
The function returns $H_n$.

In [ ]:
function hashString(str: string): number {
    let hash = 0x811c9dc5;
    for (let i = 0; i < str.length; i++) {
        hash ^= str.charCodeAt(i);
        hash = Math.imul(hash, 0x01000193);
    }
    return hash >>> 0;
}

**Example:**
For the input `str = "hello"`, the function iterates through the ASCII codes of 'h', 'e', 'l', 'l', 'o', continuously updating the hash state.
The result is the integer `1335831723`, which serves as a unique numeric fingerprint for the string "hello".

In [ ]:
const hashVal = hashString("hello");
console.log(`Hash of "hello": ${hashVal}`); // Expected: 1335831723

## Universal Comparison Function

The function `compare` defines a total ordering $\prec$ over the domain of all possible values (primitives, arrays, and sets). This ordering is required to maintain the **canonical form** (sorted order) of elements within a set.

**Input:**
*   `a`, `b`: Two values of unknown type.

**Output:**
*   Returns $< 0$ if $a \prec b$.
*   Returns $0$ if $a = b$ (structural equality).
*   Returns $> 0$ if $b \prec a$.

**Mathematical Sketch:**
The order is defined lexicographically and recursively:
1.  **Arrays**: If $a = [x_1, \dots, x_n]$ and $b = [y_1, \dots, y_m]$, compare elements $x_i$ and $y_i$ sequentially. The first non-zero difference determines the order. If one array is a prefix of the other, the shorter one is smaller.
2.  **RecursiveSets**: If $a, b \in \text{RecursiveSet}$, delegate to `a.compare(b)`.
3.  **Type Mismatch**: If types differ, an arbitrary deterministic order is applied (e.g., based on type checking).
4.  **Primitives**: Standard comparison (e.g., string or number comparison). 

The compare function is in the same cell as RecursieSet as the both depend on each other.

## RecursiveSet Definition

We define the class `RecursiveSet` and an accompanying `interface` to allow for prototype-based method assignment in subsequent cells.

The constructor initializes the set. If elements are provided, they are sorted using `compare`, and duplicates are removed via `_unique` to establish the canonical form.

$$ S = \{e_1, e_2, \dots, e_n\} \quad \text{where} \quad e_1 \prec e_2 \prec \dots \prec e_n $$

In [ ]:
function compare(a: unknown, b: unknown): number {
    if (a === b) return 0;

    if (Array.isArray(a) && Array.isArray(b)) {
        if (a.length !== b.length) return a.length - b.length;
        for (let i = 0; i < a.length; i++) {
            const diff = compare(a[i], b[i]);
            if (diff !== 0) return diff;
        }
        return 0;
    }
        
    const isSetA = a instanceof RecursiveSet;
    const isSetB = b instanceof RecursiveSet;

    if (isSetA && isSetB) {
        return a.compare(b);
    }
    
    if (isSetA !== isSetB) return isSetA ? 1 : -1;

    return (a as string | number) < (b as string | number ) ? -1 : 1;
}

class RecursiveSet<T> {
    public _elements: T[];
    public _hashCode: number | null = null;
    constructor(...elements: T[]) {
        if (elements.length === 0) {
            this._elements = [];
            this._hashCode = 0;
        } else {
            this._elements = elements.slice().sort(compare);
            this._unique();
        }
    }
}

interface RecursiveSet<T> {
    _elements: T[];
    _hashCode: number | null;
    _unique(): void;
    getHashCode(): number;
    compare(other: RecursiveSet<T>): number;
    get size(): number; // Getter property
    isEmpty(): boolean;
    has(element: T): boolean;
    add(element: T): this;
    remove(element: T): this;
    clone(): RecursiveSet<T>;
    union(other: RecursiveSet<T>): RecursiveSet<T>;
    getHash(): string;
    toString(): string;
    [Symbol.iterator](): Iterator<T>;
}

**Example for external compare function:**
Comparing arrays `['a', 'b']` and `['a', 'c']`:
1. The first elements `'a'` and `'a'` are equal.
2. The second elements `'b'` and `'c'` are compared. Since `'b' < 'c'`, the function returns a negative number (e.g., -1).
This establishes that `['a', 'b']` comes "before" `['a', 'c']`.


In [ ]:
const arr1 = ['a', 'b'];
const arr2 = ['a', 'c'];
const comparisonResult = compare(arr1, arr2);

console.log(`Compare result: ${comparisonResult}`); // Expected: -1 

**Example for constructor:**
Calling `new RecursiveSet(3, 1, 2, 2)` creates a set.
Internally, the constructor sorts the inputs to `[1, 2, 2, 3]` and then calls `_unique` to remove the duplicate `2`.
The final stored elements are `[1, 2, 3]`.


## Unique Enforcement

The private method `_unique` removes duplicate elements from the internally sorted `_elements` array in-place.

**Algorithmic Sketch:**
Given a sorted array $A$ of length $N$:
Iterate with a read pointer $r$ from $1$ to $N-1$ and a write pointer $w = 1$.
If $A[r] \neq A[r-1]$ (using `compare`), set $A[w] = A[r]$ and increment $w$.
Finally, truncate $A$ to length $w$.
This operates in $O(N)$ time.


In [ ]:
RecursiveSet.prototype._unique = function<T>(this: RecursiveSet<T>): void {
    if (this._elements.length < 2) return;
    let writeIdx = 1;
    for (let readIdx = 1; readIdx < this._elements.length; readIdx++) {
        if (compare(    this._elements[readIdx], this._elements[readIdx - 1]) !== 0) {
            this._elements[writeIdx++] = this._elements[readIdx];
        }
    }
    this._elements.length = writeIdx;
};

**Example:**
We manually simulate the state before `_unique` is called.
If `_elements` is `['a', 'b', 'b', 'c']` (sorted but with duplicates):
The method effectively compacts it to `['a', 'b', 'c']`.


In [ ]:
// Demonstration via public constructor (which calls _unique internally)
const setUniqueDemo = new RecursiveSet('a', 'b', 'b', 'c');
console.log(`Unique elements: ${setUniqueDemo._elements}`); 
// Expected output: a,b,c

## Hash Calculation

The method `getHashCode` computes a 32-bit integer hash for the set. This hash is order-dependent on the internal `_elements` array, which is valid because the array is always sorted (canonical).

**Mathematical Sketch:**
$$ H(S) = \sum_{e \in S} (31 \cdot h_{\text{prev}} + \text{hash}(e)) $$
Where $\text{hash}(e)$ is the hash of the element $e$. If $e$ is itself a `RecursiveSet`, its `getHashCode` is called recursively.


In [ ]:
RecursiveSet.prototype.getHashCode = function<T>(this: RecursiveSet<T>): number {
    if (this._hashCode !== null) return this._hashCode;
    let h = 0;
    for (const el of this._elements) {
        let elHash = 0;
        if (el instanceof RecursiveSet) {
            elHash = el.getHashCode();
        } else if (typeof el === 'string') {
            elHash = hashString(el);
        } else {
            elHash = hashString(String(el));
        }
        h = Math.imul(31, h) + elHash;
    }
    this._hashCode = h | 0;
    return this._hashCode;
};

**Example:**
For a set `{ 'a', 'b' }`:
1. Computes hash of 'a' and updates the accumulator `h`.
2. Computes hash of 'b' and mixes it into `h`.
The final `h` is stored in `_hashCode` so subsequent calls return immediately without recalculation.

In [ ]:
const setHashDemo = new RecursiveSet('a', 'b');
const hCode = setHashDemo.getHashCode();
console.log(`HashCode: ${hCode}`); // Expected output: -2071647687

## Set Comparison

The method `compare` compares this set with `other`. It is optimized using the cached hash code.

**Optimization Strategy:**
1.  **Hash Check**: If $H(A) \neq H(B)$, the sets are strictly unequal. Return order based on hash values. $O(1)$.
2.  **Cardinality Check**: If $|A| \neq |B|$, return difference in size. $O(1)$.
3.  **Element Check**: Iterate through sorted elements. If $A[i] \neq B[i]$, return the difference. $O(N)$.


In [ ]:
RecursiveSet.prototype.compare = function<T>(this: RecursiveSet<T>, other: RecursiveSet<T>): number {
    if (this === other) return 0;
    const h1 = this.getHashCode();
    const h2 = other.getHashCode();
    if (h1 !== h2) return h1 < h2 ? -1 : 1;
    const len = this._elements.length;
    if (len !== other._elements.length) return len - other._elements.length;
    for (let i = 0; i < len; i++) {
        const cmp = compare(this._elements[i], other._elements[i]);
        if (cmp !== 0) return cmp;
    }
    return 0;
};

**Example:**
Comparing Set A `{1, 2}` and Set B `{1, 3}`:

1.  **Hash Check**: The first step is comparing the hash codes.
    *   If $H(A) \neq H(B)$, the order is determined solely by the hashes (return $H(A) - H(B)$).
    *   Since the hash calculation uses FNV-1a, $H(\{1, 2\}) \approx \text{3.5B}$ and $H(\{1, 3\}) \approx \text{1.6B}$.
    *   Because $H(A) > H(B)$, the function returns a positive value (e.g., `1`), effectively saying $A > B$ in the canonical ordering.
2.  **Element Check**: This fallback only happens if hashes are identical (collision). In that case, it would compare elements: `1 == 1`, but `2 < 3`, yielding `-1`.

In this specific case, the **Hash Check** dominates.

In [ ]:
const setA = new RecursiveSet(1, 2);
const setB = new RecursiveSet(1, 3);
console.log(`Set Compare (1,2 vs 1,3): ${setA.compare(setB)}`); // Expected: 1

## Size Property

Returns the cardinality of the set $|S|$.

In [ ]:
Object.defineProperty(RecursiveSet.prototype, "size", {
    get: function<T>(this: RecursiveSet<T>) {
        return this._elements.length;
    },
    enumerable: false,
    configurable: true
});

**Example:**
If the internal array is `['x', 'y', 'z']`, accessing `mySet.size` returns `3`.


In [ ]:
const setSizeDemo = new RecursiveSet('x', 'y', 'z');
console.log(`Size: ${setSizeDemo.size}`); // Expected: 3

## Empty Check

Returns `true` if $S = \emptyset$.


In [ ]:
RecursiveSet.prototype.isEmpty = function<T>(this: RecursiveSet<T>): boolean {
    return this._elements.length === 0; 
};

**Example:**
`new RecursiveSet().isEmpty()` returns `true`.
`new RecursiveSet(1).isEmpty()` returns `false`.

In [ ]:
console.log(`Empty set: ${new RecursiveSet().isEmpty()}`); // true
console.log(`Non-empty set: ${new RecursiveSet(1).isEmpty()}`); // false

## Membership Check

The method `has` checks if an element $e$ is contained in the set $S$.
$$ \text{has}(e) \iff \exists x \in S : x = e $$
Due to the internal structure, this performs a linear scan using the structural `compare` function.


In [ ]:
RecursiveSet.prototype.has = function<T>(this: RecursiveSet<T>, element: T): boolean {
    for (let i = 0; i < this._elements.length; i++) {
        if (compare(this._elements[i], element) === 0) return true;
    }
    return false;
};

**Example:**
Given set `S = {[1], [2]}` (arrays as elements).
Calling `S.has([1])` creates a new array `[1]`, compares it with the existing `[1]` using deep comparison. Since they are structurally equal, it returns `true`.

In [ ]:
const setHasDemo = new RecursiveSet([1], [2]);
console.log(`Has [1]: ${setHasDemo.has([1])}`); // Expected: true

## Add Element

Mutates the set by adding an element $e$.
$$ S' = S \cup \{e\} $$

**Algorithm:**
1.  Check if $e$ can be appended to the end (optimization for sorted insertions).
2.  Otherwise, perform an insertion sort step to place $e$ at the correct index $i$ such that $e_{i-1} \prec e \prec e_{i}$.
3.  Invalidates `_hashCode` to trigger re-computation on next access.

In [ ]:
RecursiveSet.prototype.add = function<T>(this: RecursiveSet<T>, element: T): RecursiveSet<T> {
    const len = this._elements.length;
    if (len > 0) {
        const last = this._elements[len - 1];
        const cmp = compare(last, element);
        if (cmp < 0) {
            this._elements.push(element);
            this._hashCode = null;
            return this;
        }
        if (cmp === 0) return this;
    } else {
        this._elements.push(element);
        this._hashCode = null;
        return this;
    }
    for (let i = 0; i < len; i++) {
        const cmp = compare(this._elements[i], element);
        if (cmp === 0) return this;
        if (cmp > 0) {
            this._elements.splice(i, 0, element);
            this._hashCode = null;
            return this;
        }
    }
    return this;
};

**Example:**
Given set `S = {1, 3}`.
Calling `S.add(2)` finds that `1 < 2 < 3`. It inserts `2` at index 1.
The internal state becomes `{1, 2, 3}` and the hash is marked as invalid (null).


In [ ]:
const setAddDemo = new RecursiveSet(1, 3);
setAddDemo.add(2);
console.log(`After adding 2: ${setAddDemo._elements}`); // Expected: 1, 2, 3

## Remove Element

Mutates the set by removing an element $e$.
$$ S' = S \setminus \{e\} $$
Finds the index of $e$ and splices it out. Invalidates `_hashCode`.

In [ ]:
RecursiveSet.prototype.remove = function<T>(this: RecursiveSet<T>, element: T): RecursiveSet<T> {
    for (let i = 0; i < this._elements.length; i++) {
        if (compare(this._elements[i], element) === 0) {
            this._elements.splice(i, 1);
            this._hashCode = null;
            return this;
        }
    }
    return this;
};

**Example:**
Given set `S = {1, 2, 3}`.
Calling `S.remove(2)` finds `2` at index 1 and removes it.
The internal state becomes `{1, 3}`.


In [ ]:
const setRemoveDemo = new RecursiveSet(1, 2, 3);
setRemoveDemo.remove(2);
console.log(`After removing 2: ${setRemoveDemo._elements}`); // Expected: 1, 3

## Clone

Creates a shallow copy of the set structure.
$$ S_{\text{copy}} = S $$
Since the internal array `_elements` is cloned but elements themselves are not deep-copied (unless they are primitives), this is efficient. The cached hash code is preserved.

In [ ]:
RecursiveSet.prototype.clone = function<T>(this: RecursiveSet<T>): RecursiveSet<T> {
    const s = new RecursiveSet<T>();
    s._elements = this._elements.slice();
    s._hashCode = this._hashCode;
    return s;
};

**Example:**
Given `S1 = {1, 2}`.
`const S2 = S1.clone()` creates `S2`.
Changing `S1` (e.g., adding 3) does not affect `S2`. `S2` remains `{1, 2}`.


In [ ]:
const s1Clone = new RecursiveSet(1, 2);
const s2Clone = s1Clone.clone();
s1Clone.add(3);
console.log(`S1: ${s1Clone._elements} | S2: ${s2Clone._elements}`); 
// Expected: S1: 1,2,3 | S2: 1,2

## Union

Computes the union of two sets.
$$ R = A \cup B = \{ x \mid x \in A \lor x \in B \} $$

**Algorithm:**
Since both $A$ and $B$ are sorted internally, the union is computed using a **merge algorithm** (similar to MergeSort) in $O(|A| + |B|)$ time. Pointers $i$ and $j$ traverse $A$ and $B$ respectively, always picking the smaller element to append to $R$.

In [ ]:
RecursiveSet.prototype.union = function<T>(this: RecursiveSet<T>, other: RecursiveSet<T>): RecursiveSet<T> {
    const s = new RecursiveSet<T>();
    const arrA = this._elements;
    const arrB = other._elements;
    const res: T[] = [];  
    let i = 0, j = 0;
    while (i < arrA.length && j < arrB.length) {
        const cmp = compare(arrA[i], arrB[j]);
        if (cmp < 0) { res.push(arrA[i]); i++; }
        else if (cmp > 0) { res.push(arrB[j]); j++; }
        else { res.push(arrA[i]); i++; j++; }
    }
    while (i < arrA.length) res.push(arrA[i++]);
    while (j < arrB.length) res.push(arrB[j++]);
    
    s._elements = res;
    return s;
};

**Example:**
Given `A = {1, 3}` and `B = {2, 3}`.
1. Compare `1` and `2`: `1` is smaller, added to result.
2. Compare `3` and `2`: `2` is smaller, added to result.
3. Compare `3` and `3`: Equal, `3` added once, both pointers advance.
Result: `{1, 2, 3}`.

In [ ]:
const setUnionA = new RecursiveSet(1, 3);
const setUnionB = new RecursiveSet(2, 3);
const setUnionRes = setUnionA.union(setUnionB);
console.log(`Union: ${setUnionRes._elements}`); // Expected: 1, 2, 3

## String Representation

Formats the set as a string for display. Recursively formats elements.
Examples: `∅`, `{1, 2}`, `{{'a'}, {'b'}}`.

In [ ]:
RecursiveSet.prototype.toString = function<T>(this: RecursiveSet<T>): string {
    if (this.isEmpty()) return "∅";   
    const elementsStr = this._elements.map(el => {
        if (Array.isArray(el)) {
            return `[${el.map(x => typeof x === 'string' ? `'${x}'` : x).join(', ')}]`;
        }
        return String(el);
    });
    return `{${elementsStr.join(', ')}}`;
};


RecursiveSet.prototype[Symbol.for('nodejs.util.inspect.custom')] = function<T>(this: RecursiveSet<T>): string { 
    return this.toString(); 
};

**Example:**
Given `S = {1, {2}}`.
Calling `S.toString()` produces the string `"{1, {2}}"`, which visually represents the nested structure.

In [ ]:
const subSetStr = new RecursiveSet(2);
const setStrDemo = new RecursiveSet<number | RecursiveSet<number>>(1, subSetStr);
console.log(`String: ${setStrDemo.toString()}`); // Expected: {1, {2}}

## Get Hash String

The method `getHash` provides a string representation of the set's hash code. 

In [ ]:
RecursiveSet.prototype.getHash = function<T>(this: RecursiveSet<T>): string { 
    return String(this.getHashCode()); 
};

**Example:**
If `getHashCode()` returns the integer `873244444`, `getHash()` returns the string `"873244444"`.

In [ ]:
const setGetHashDemo = new RecursiveSet(1);
console.log(`Hash String: "${setGetHashDemo.getHash()}"`);

## Node.js Custom Inspector

This method implements the symbol `nodejs.util.inspect.custom`. It allows the Node.js runtime (and by extension, Jupyter kernels based on Node.js) to print a human-readable string representation of the `RecursiveSet` when logged to the console, rather than displaying the internal object structure.

**Behavior:**
*   Delegates directly to `this.toString()`.

In [ ]:
RecursiveSet.prototype[Symbol.for('nodejs.util.inspect.custom')] = function<T>(this: RecursiveSet<T>): string { 
    return this.toString(); 
};

**Example:**
When executing `console.log(mySet)` in Node.js, this method is called automatically. Instead of printing `RecursiveSet { _elements: [...] }`, it prints `{a, b, c}`.

In [ ]:
const setInspectDemo = new RecursiveSet('a', 'b', 'c');
setInspectDemo) // Expected: {a, b, c}

## Iterator

Enables usage of `RecursiveSet` in `for...of` loops and spread syntax.


In [ ]:
RecursiveSet.prototype[Symbol.iterator] = function<T>(this: RecursiveSet<T>): Iterator<T> {
    return this._elements[Symbol.iterator]();
};

**Example:**
Using `for (const x of mySet) { console.log(x); }` works because of this method.
Similarly, `[...mySet]` converts the set back into an array using this iterator.


In [ ]:
const setIterDemo = new RecursiveSet(1, 2, 3);
const asArray = [...setIterDemo];
console.log(`Spread to Array: ${JSON.stringify(asArray)}`); // Expected: [1,2,3]

## Helper: From Iterable

Creates a `RecursiveSet` from any iterable (e.g., Array, Set, Generator).

In [ ]:
function fromIterable<T>(iterable: Iterable<T>): RecursiveSet<T> {
    return new RecursiveSet(...iterable);
}

**Example:**
Given a native JS Set `nativeSet = new Set([1, 2])`.
`fromIterable(nativeSet)` creates a `RecursiveSet` containing `1` and `2`.


In [ ]:
const nativeSet = new Set([1, 2]);
const recSetFromNative = fromIterable(nativeSet);
console.log(`From Native Set: ${recSetFromNative}`); // Expected: {1, 2}